In [ ]:
from dask.distributed import Client
import dask.dataframe as ddf

client = Client(“…”)

# Lazy-load a large dataset of climate simulations from shared filesystem
df = ddf.read_parquet("run_24-06.parquet")

# Lazy preprocessing 
df = df[df["temperature"].notnull()]
df = df.assign(anomaly = df["temperature”]-
     df["temperature_baseline"])

In [ ]:
# Group by region and day and compute 
# summary statistics across all worker nodes
daily_stats = (df.groupby(["region","day"])
              .agg({"anomaly":["mean","std"],
               "precipitation":"sum"})
               .persist())

# Compute risk score
def risk_score(row):
    return 1.0 row[("anomaly", "mean")] + 0.1
    *row[("precipitation", "sum")]

daily_stats = daily_stats.map_partitions(
              lambda part: part.assign(
              risk_score = part.apply(
              risk_score, axis=1)))

In [ ]:
# Run distributed computation and get a subset of the result 
high_risk = daily_stats[daily_stats
            ["risk_score"] > 10.0]

# large-scale distributed execution
result = high_risk.compute()

# Save summary for analysis and visualization
result.to_parquet("high_risk_24-06.parquet") 